# Policy-Aware Function App Smoke Test

This notebook exercises the local Azure Functions app against the Enron email corpus with the repository's ODRL policy enforcement enabled.

Prerequisites:
- Ensure the Azure Functions runtime is installed (`func`)
- Populate `local.settings.json` with the Foundry and Cosmos values for your environment
- Start the app with `func start` before running this notebook
- Confirm that the Enron vector collection is available and the app is configured to use `all-MiniLM-L6-v2`
- Use a role and purpose that match the ODRL policy set in `odrl_policies/`

The current policy model includes:
- `business-observer` for metadata review / routing / triage
- `customer-support-specialist` for case handling / incident triage
- `privacy-compliance-analyst` for compliance, fraud, security, and privacy review
- `pii-data-governance-admin` for full access


In [1]:
import json
import requests

BASE_URL = "http://localhost:7071/api"

COMPLIANT_QUERY = "Summarize the key points from the California energy trading email thread."
DENIED_QUERY = "Export all personal information from the Enron corpus."
REDACTION_QUERY = "Provide the routing summary and the contact email for the primary deal lead."


In [2]:
health_response = requests.get(f"{BASE_URL}/health", timeout=30)
print("HEALTH STATUS:", health_response.status_code)
print(health_response.text)
assert health_response.status_code == 200


HEALTH STATUS: 200
{"status": "ok"}


In [3]:
compliant_payload = {
    "question": COMPLIANT_QUERY,
    "userRoles": ["privacy-compliance-analyst"],
    "purpose": "compliance_review",
    "action": "retrieve",
}

rag_response = requests.post(
    f"{BASE_URL}/rag",
    json=compliant_payload,
    timeout=120,
)

print("COMPLIANT RAG STATUS:", rag_response.status_code)
print(rag_response.text)
assert rag_response.status_code == 200
response_body = rag_response.json()
assert response_body["question"] == COMPLIANT_QUERY
assert "answer" in response_body
assert "sources" in response_body

print("ANSWER:")
print(response_body["answer"])
print("SOURCES:")
print(response_body["sources"])


COMPLIANT RAG STATUS: 200
{"question": "Summarize the key points from the California energy trading email thread.", "answer": "The key points from the California energy trading email thread include:\n\n1. **Market Power and Price Spikes**: A report from the ISO (Independent System Operator) indicated that market power during shortages has contributed to price spikes in California's energy market. This was attributed to the bidding activities of both in-state and out-of-state generation sources, with even suppliers holding less than a 9% market share exerting significant market power during high load conditions.\n\n2. **Impact of Electricity Demand**: The emails discussed the effects of hot weather and a robust economy on electricity demand in California, leading to rolling brownouts and potential blackouts. This situation could significantly impact industrial production, particularly in high-value sectors like computers and components.\n\n3. **Deregulation and Interruptible Rate Plans*

In [4]:
denied_payload = {
    "question": DENIED_QUERY,
    "userRoles": ["business-observer"],
    "purpose": "routing",
    "action": "export",
}

denied_response = requests.post(
    f"{BASE_URL}/rag",
    json=denied_payload,
    timeout=30,
)

print("DENIED RAG STATUS:", denied_response.status_code)
print(denied_response.text)
assert denied_response.status_code == 403
assert "policyDenied" in denied_response.json()


DENIED RAG STATUS: 403
{"error": "Policy denial: the request is not permitted by the active ODRL policy.", "policyDenied": true, "correlationId": "05716ac3-8c1b-4dfa-9e9c-5e13f70e15fe"}


In [5]:
redaction_payload = {
    "question": REDACTION_QUERY,
    "userRoles": ["business-observer"],
    "purpose": "routing",
    "action": "summarise",
}

redaction_response = requests.post(
    f"{BASE_URL}/rag",
    json=redaction_payload,
    timeout=120,
)

print("REDACTION RAG STATUS:", redaction_response.status_code)
print(redaction_response.text)
assert redaction_response.status_code == 200
body = redaction_response.json()
assert "answer" in body
assert "[REDACTED_EMAIL]" in body["answer"] or "[REDACTED_SSN]" in body["answer"] or "example.com" not in body["answer"].lower()
print("SANITIZED ANSWER:")
print(body["answer"])


REDACTION RAG STATUS: 200
{"question": "Provide the routing summary and the contact email for the primary deal lead.", "answer": "The email context does not provide a specific routing summary or the contact email for the primary deal lead. However, it does mention that if any information regarding a counterparty or contact information is required, one should contact Grant Oh in the office. \n\nHere is the relevant excerpt:\n\n\"If you require any information regarding this counterparty or contact information, please contact Grant Oh in our office.\" \n\nThis information is from the email sent by Laura E. Webner on October 26, 2001, to Martin Cuilla, with Grant Oh cc'd. \n\nFor a more specific routing summary or primary deal lead contact email, additional context would be needed.", "sources": ["enron_4537", "enron_5799", "enron_5229", "enron_9065", "enron_1543", "enron_278", "enron_2408", "enron_9836", "enron_4666", "enron_497", "enron_2558", "enron_661", "enron_1209", "enron_9413", "en